In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import * 
from e_2_CVAE import *


# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'van' # van or barr
model_type = 'bs' # hes or bs

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

# cvae training settings
if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942


In [ ]:
# training1
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 1024*7 #
n_epochs    = 200 # loss 수렴할 때까지 
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0

n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}_{batch_size//7}*7.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_parallel(
    model_type, barr_type, dim_z, hidden_dims, batch_size, n_epochs, lr, beta, save_path,
    gpu_ids=[0, 1, 2, 3, 4, 5, 6],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


if barr_type == 'barr':
    compare_prices(cvae, B, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples) 
elif barr_type == 'van':
    compare_prices(cvae, None, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples)


plt.plot(loss_history['recon_loss'], label='Recon')
plt.plot(loss_history['KL_loss'],    label='KL')
plt.plot(loss_history['total_loss'], label='Total')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# training2
dim_z       = 4 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 1024*7 #
n_epochs    = 200 # loss 수렴할 때까지 
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0

n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}_{batch_size//7}*7.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_parallel(
    model_type, barr_type, dim_z, hidden_dims, batch_size, n_epochs, lr, beta, save_path,
    gpu_ids=[0, 1, 2, 3, 4, 5, 6],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


if barr_type == 'barr':
    compare_prices(cvae, B, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples) 
elif barr_type == 'van':
    compare_prices(cvae, None, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples)


plt.plot(loss_history['recon_loss'], label='Recon')
plt.plot(loss_history['KL_loss'],    label='KL')
plt.plot(loss_history['total_loss'], label='Total')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()